In [ ]:
def hamming_distance(genome1, genome2):
    if len(genome1) != len(genome2):
        raise ValueError("Genomes must be of the same length")
    
    distance = 0
    for base1, base2 in zip(genome1, genome2):
        if base1 != base2:
            distance += 1
            
    return distance

In [ ]:
def identify_kmer_mismatches(genome1, genome2, mismatch_value):

    mismatches = []
    for i in range(len(genome2) - len(genome1) + 1):
        kmer1 = genome1
        kmer2 = genome2[i:i+len(genome1)]
        
        if hamming_distance(kmer1, kmer2) <= int(mismatch_value):
            mismatches.append((i))
    
    return mismatches

In [ ]:
def immediate_neighbours(pattern, hamming_distance_threshold):
    
    neighborhood = set()
    if hamming_distance_threshold == 0:
        return {pattern}
    
    for i in range(len(pattern)):
        for nucleotide in "ACGT":
            neighbor = pattern[:i] + nucleotide + pattern[i+1:]
            neighborhood.update(immediate_neighbours(neighbor, hamming_distance_threshold - 1))
    
    return neighborhood

In [ ]:
def motif_enumeration(dna, k, d):
    sequences = dna.split()
    intersection = set()
    for sequence in sequences:
        patterns = set()
        
        print(f"Processing sequence: {sequence}")
        for i in range(len(sequence) - k + 1):
            kmer = sequence[i:i+k]
            neighbors = immediate_neighbours(kmer, d)
            patterns.update(neighbors)
        if not intersection:
            intersection = patterns
        else:
            intersection = intersection.intersection(patterns)
            
        
    return intersection

In [ ]:
dna = "TCCGGTTGTTCATTAATGATGTTGA CATTTTGAACATACCCGGTCCCCTC GAGTCTTGTTTTACGCATTACCCTC TTCTCCCGACCTTTTCTGTGCTTTA TGGATGTTTAAGGACGACATCCTTG AACAATAACCCGGAGCTTTGAGGGG"
k = 5
d = 2
print(motif_enumeration(dna, k, d))

In [26]:
motif = """TCGGGGGTTTTT
CCGGTGACTTAC
ACGGGGATTTTC
TTGGGGACTTTT
AAGGGGACTTCC
TTGGGGACTTCC
TCGGGGATTCAT
TCGGGGATTCCT
TAGGGGAACTAC
TCGGGTATAACC
"""

In [31]:
def entropy_calculation(motif):
    from math import log2
    
    motif = motif.split()
    count_of_motifs = len(motif)
    column_motif = ["".join([motif[i][j] for i in range(count_of_motifs)]) for j in range(len(motif[0]))]
    entropy = 0.0
    
    for column in column_motif:
        count_a = column.count("A")/count_of_motifs
        count_c = column.count("C")/count_of_motifs
        count_g = column.count("G")/count_of_motifs
        count_t = column.count("T")/count_of_motifs
        
        entropy -= count_a * log2(count_a) if count_a > 0 else 0
        entropy -= count_c * log2(count_c) if count_c > 0 else 0
        entropy -= count_g * log2(count_g) if count_g > 0 else 0
        entropy -= count_t * log2(count_t) if count_t > 0 else 0
        
    # for nucleotide in "ACGT":
    #     if count[nucleotide] > 0:
    #         probability = count[nucleotide] / total
    #         entropy -= probability * log2(probability)
    
    return entropy

In [32]:
entropy_calculation(motif)

9.916290005356972

In [49]:
def motif(pattern, dna):
    
    length_of_pattern = len(pattern)
    motifs = []
    distances = []
    for seq in dna:
        min_distance = float('inf')
        for i in range(len(seq) - length_of_pattern + 1):
            kmer = seq[i:i+length_of_pattern]
            distance = hamming_distance(pattern, kmer)
            if distance < min_distance:
                min_distance = distance
                best_kmer = kmer
        motifs.append(best_kmer)
        distances.append(min_distance)
                
    
    return sum(distances)

In [50]:
pattern = "GATTACA"
dna = [
    "CGTACGATTACACGAT",
    "GATTACAGTCAGATTG",
    "GATCGATTGACGATTC",
    "CGATTACACGTACGAT"
]

# Find motifs
result = motif(pattern, dna)
print(result)

3


In [51]:
def create_possible_kmers(k):
    from itertools import product
    return [''.join(p) for p in product('ACGT', repeat=k)]

In [52]:
def median_string(dna, k):
    min_distance = float('inf')
    all_patterns = create_possible_kmers(k)
    
    for pattern in all_patterns:
        distance = motif(pattern, dna)
        if distance < min_distance:
            min_distance = distance
            median = pattern
            
    return median

In [53]:
dna = ["AAATTGACGCAT", "GACGACCACGTT", "CGTCAGCGCCTG", "GCTGAGCACCGG", "AGTTCGGGACAG"]
k = 3
print(median_string(dna, k))

GAC


In [54]:
dna = ["AGGCAGACTAAATCCGGAGACCCGGCTGTGCATGAGAGATCA",
"TAACATCCGCCGATCGACACTAAAGTGGATAATTTCAGCGGT"
"ACTCAATCGTCGCGTTGTCTCGTCCCGCGACCCGGTGTGGCC",
"CAAACCAAGGGGTAAAGGGAATGAACTCCTACTTAATGGTCC",
"CTTTTGACTAAACCAGGGTAGCTTGCCTTGAGTTGTCTACGT",
"TCACTCACTAAATCGACGGCCTATTGCAGCGAATGCTCGTAT",
"ATCAGGCATTCAATTATTCTCGCTTCCGTCACTTAACCCCGA",
"GCCTCTGCGATGGTCAATTCACATTCGAATACTTAAAAATTC",
"CATAGTGGCTACACTTAAATCTGGATGTGGCGCAAAGAAGAT",
"GTAGGCTTGGACACTGAAGCATCGAACTATAGGTCTTTAGGG"]

k = 6
print(median_string(dna, k))


ACTAAA


In [ ]:
def profile_most_probable_kmer_search